# Day 33 — 올바른 검증: GroupKFold와 데이터 누수

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-5/day-33.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-5/day-33.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 같은 데이터, 같은 모델, 나누는 방법만 바꾸기

In [ ]:
import numpy as np
import pandas as pd

FEATURES = ["stride_time_cv", "accel_rms", "tremor_band_power",
            "jerk_mean", "step_regularity", "arm_swing_asym"]

def make_demo_table(seed=42, n_participants=50, n_repeats=6):
    """연습용 가상 feature table. 실제 환자 데이터가 아니다."""
    rng = np.random.default_rng(seed)
    pd_shift = np.array([0.6, -0.5, 0.6, 0.2, -0.4, 0.3])   # PD군 평균이 이동하는 정도
    rows = []
    for i in range(n_participants):
        group = "PD" if i < n_participants // 2 else "HC"
        age = rng.normal(68 if group == "PD" else 62, 7)
        sex = rng.choice(["F", "M"])
        person = rng.normal(0, 1.0, size=len(FEATURES))      # 그 사람 고유의 '움직임 지문'
        for rep in range(1, n_repeats + 1):
            values = person + rng.normal(0, 0.25, size=len(FEATURES))  # 측정마다 생기는 흔들림
            if group == "PD":
                values = values + pd_shift
            values[0] += 0.06 * (age - 65)                   # stride_time_cv는 나이와도 관련
            rows.append({"participant_id": f"P{i + 1:03d}", "group": group,
                         "age": round(age, 1), "sex": sex, "repeat": rep,
                         **dict(zip(FEATURES, values.round(3)))})
    return pd.DataFrame(rows)

df = make_demo_table()

# 🔁 실제 데이터로 바꿀 지점 ---------------------------------------------
# 4주차(Day 26)에 만든 feature table이 있으면 위의 df = make_demo_table() 대신:
# df = pd.read_csv("data/physionet_pd/derived/pd_features_records.csv")   # 내 파일 경로로
# FEATURES = ["내_feature_1", "내_feature_2"]                # 내 feature 열 이름으로
# 그리고 아래 두 이름을 내 표의 열 이름에 맞춘다.
ID_COL = "participant_id"   # 참가자 ID 열
GROUP_COL = "group"         # 진단군 열 (이 예시에서는 "PD" / "HC")
# -------------------------------------------------------------------------

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = df[FEATURES]
y = (df[GROUP_COL] == "PD").astype(int)       # 🔁 내 군 이름으로
groups = df[ID_COL]

models = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression()),
    "random forest": RandomForestClassifier(n_estimators=200, random_state=0),
}
row_cv = KFold(n_splits=5, shuffle=True, random_state=0)   # ❌ 행 단위 무작위 분할
group_cv = GroupKFold(n_splits=5)                          # ✅ 참가자 단위 분할

print(f"{'model':<22}{'row-wise (leaky)':>18}{'participant-wise':>18}{'inflation':>11}")
for name, model in models.items():
    leaky = cross_val_score(model, X, y, cv=row_cv).mean()
    honest = cross_val_score(model, X, y, cv=group_cv, groups=groups).mean()
    print(f"{name:<22}{leaky:>18.2f}{honest:>18.2f}{leaky - honest:>+11.2f}")

### 예시 2 — 모델이 정말 "사람"을 외우는지 확인하기: 가짜 정답 실험

정말 개인을 외워서 생긴 일인지 확인하는 실험입니다. 참가자 50명에게 **진단과 무관한 가짜 군**을 무작위로 붙입니다. 이제 feature와 정답 사이에는 아무 관계가 없으므로, 정직한 평가라면 정확도는 동전 던지기(0.5) 근처여야 합니다. 예시 1에 이어서 실행합니다.

In [ ]:
# 첫 번째 fold에서 평가용 행의 참가자가 학습용에도 있는지 세어 보기
first_folds = {"row-wise": next(row_cv.split(X, y)),
               "participant-wise": next(group_cv.split(X, y, groups))}
for label, (train_idx, test_idx) in first_folds.items():
    seen = groups.iloc[test_idx].isin(set(groups.iloc[train_idx]))
    print(f"{label:<17} 평가용 {len(test_idx)}행 중 학습용에 같은 참가자가 있는 행: {seen.sum()}")

# 참가자마다 가짜 군(0/1)을 무작위로 배정: 진짜 신호가 전혀 없는 정답
rng = np.random.default_rng(1)
people = df[ID_COL].unique()
fake_label = dict(zip(people, rng.permutation(len(people)) % 2))
y_fake = df[ID_COL].map(fake_label)

forest = models["random forest"]
leaky = cross_val_score(forest, X, y_fake, cv=row_cv).mean()
honest = cross_val_score(forest, X, y_fake, cv=group_cv, groups=groups).mean()
print()
print(f"가짜 정답 + row-wise 분할         accuracy = {leaky:.2f}")
print(f"가짜 정답 + participant-wise 분할 accuracy = {honest:.2f}")

### 예시 3 — 참가자 단위 검증 코드와 feature 누수 점검

오늘의 산출물이 될 검증 코드입니다. fold마다 겹치는 참가자가 없는지 확인하고, 평가용 인원과 PD 비율, 점수를 표로 남깁니다. 이어서 일부러 "정답에서 파생된 열"을 하나 넣어 feature 누수가 어떻게 보이는지 확인합니다. 예시 2에 이어서 실행합니다.

In [ ]:
from sklearn.base import clone

def participant_cv(model, X, y, groups, n_splits=5):
    """참가자 단위 교차검증. fold별 점검 결과와 accuracy를 표로 돌려준다."""
    rows = []
    for fold, (train_idx, test_idx) in enumerate(GroupKFold(n_splits=n_splits).split(X, y, groups), start=1):
        overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
        assert not overlap, f"fold {fold}: 참가자 누수 {overlap}"
        fitted = clone(model).fit(X.iloc[train_idx], y.iloc[train_idx])   # 전처리도 학습용에서만 fit
        rows.append({"fold": fold,
                     "test_participants": groups.iloc[test_idx].nunique(),
                     "test_PD_ratio": round(y.iloc[test_idx].mean(), 2),
                     "accuracy": round(fitted.score(X.iloc[test_idx], y.iloc[test_idx]), 2)})
    return pd.DataFrame(rows)

baseline = models["logistic regression"]
result = participant_cv(baseline, X, y, groups)
print(result.to_string(index=False))
print(f"평균 accuracy = {result['accuracy'].mean():.2f} (fold 간 표준편차 {result['accuracy'].std():.2f})")

# --- feature 누수 실험: 진단 뒤에야 생기는 정보를 feature로 넣으면? ---
rng = np.random.default_rng(2)
visits = {pid: (rng.poisson(6) + 2 if g == "PD" else rng.integers(0, 2))
          for pid, g in df.drop_duplicates(ID_COL)[[ID_COL, GROUP_COL]].values}
X_leaky = X.assign(neurology_visits=df[ID_COL].map(visits))   # 가상의 '신경과 방문 횟수'

leaky_result = participant_cv(baseline, X_leaky, y, groups)
print()
print(f"누수 열을 넣은 뒤 평균 accuracy = {leaky_result['accuracy'].mean():.2f}")

# 점검: feature 하나만으로 두 군이 거의 완벽히 갈리는 열 찾기 (참가자 단위)
from sklearn.metrics import roc_auc_score
person = X_leaky.groupby(groups).mean()
person_y = y.groupby(groups).first()
auc = person.apply(lambda col: roc_auc_score(person_y, col))
auc = auc.where(auc >= 0.5, 1 - auc).sort_values(ascending=False)   # 방향과 무관하게 0.5~1로
print()
print("feature 하나만 썼을 때의 구분 정도 (1.0 = 완벽히 갈림):")
print(auc.round(2))